In [1]:
import torch
import torch.nn as nn
class reshape(nn.Module):
    '''
    reshapes the 3-order tensor into 6-order tensor

    ----------
    split : list
        split indices to be applied to each mode of the 3-order tensor

    map_type : int
        based on attached - 1 or compressed - 2 splitting method

    device : str
        operation device, default value is cpu


    inputs a 3-order torch.tensor

    returns a 6-order torch.tensor
    '''
    def __init__(self, split, map_type=1, device='cpu'):
        super(reshape, self).__init__()

        self.split = split
        self.map_type = map_type
        self.device = device

    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []

        if self.map_type == 1:
            # Approach 1 : Attached
            C_indices, H_indices, W_indices = [
                [sum(dim // self.split[i] for _ in range(j)) for j in range(self.split[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]

            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1],
                                           H_indices[j]:H_indices[j+1],
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            # Approach 2 : Compressed
            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            C_stride_indices = torch.arange(i, C, self.split[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.split[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.split[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)

        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.split[0] * self.split[1] * self.split[2])
        result = torch.cat(chunks).view(
            batch_size, self.split[0], self.split[1], self.split[2],
            *chunks[0].shape[1:])
        return result

    def forward(self, x):
        chunks = self.split_into_chunks(x)
        output = self.stack_chunks_to_form_tensor(chunks)
        return output

In [2]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 8

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [5]:
def topk_accuracy(outputs, targets, topk=(1,)):
    '''
    calculates top-k accuracy

    ----------
    outputs : torch.tensor

    targets : torch.tensor

    topk  : tuple
        calculates top k accuracy given outpurs and targets


    reutrns a python dictionary of top i <= k accuracies

    '''
    maxk = max(topk)
    _, topk_indices = torch.topk(input=outputs, k=maxk, dim=1, largest=True, sorted=True)
    correct = topk_indices.eq(targets.view(-1, 1).expand_as(topk_indices))
    accuracies = {}
    for k in topk:
        correct_k = correct[:,:k].float().sum()
        accuracies[k] = {'correct':correct_k, 'accuracy': (correct_k / outputs.shape[0]) * 100.0}
    return accuracies

In [6]:
def cp(module):
  return sum(p.numel() for p in module.parameters())

In [7]:
def print_gpu_memory_usage(stage):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    string = f'{stage} - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB'
    print(string)
    return string


In [8]:
def append_to_file(file_name, text):
    with open(file_name, 'a') as file:
        file.write(text + '\n')

# FC layers

In [9]:
class CNN1(nn.Module):
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256, bias = False)
        self.fc2 = nn.Linear(256,10, bias = False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model1 = CNN1().to(device)


In [10]:
classifier1 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256, bias = False),
    nn.Linear(256, 10, bias = False)
)

print(cp(classifier1))
append_to_file(file_name='TCL&TRL_report.txt', text=f'FC classifier # parameters {cp(classifier1)}')

1051136


In [11]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters())

In [12]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model1.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model1.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [13]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'FC took {end_time - start_time} time')
append_to_file(file_name='TCL&TRL_report.txt', text=f'FC had {string}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'FCL last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

Train epoch 1: top1=0.5359799861907959%, top2=0.7349599599838257%, top3=0.8341599702835083%, top4=0.8972599506378174%, top5=0.9363600015640259%, loss=0.161170709181726, time=7.476123094558716s
Test epoch 1: top1=0.6298999786376953%, top2=0.8109999895095825%, top3=0.8906999826431274%, top4=0.9386999607086182%, top5=0.9633999466896057%, loss=0.13056595600098372, time=0.9626123905181885s
Memory Usage  - Allocated: 32.59 MB, Reserved: 68.00 MB
Train epoch 2: top1=0.6709399819374084%, top2=0.8389399647712708%, top3=0.9112799763679504%, top4=0.950499951839447%, top5=0.9717999696731567%, loss=0.11672059639692306, time=7.16210150718689s
Test epoch 2: top1=0.6714000105857849%, top2=0.8315999507904053%, top3=0.9037999510765076%, top4=0.945099949836731%, top5=0.9681999683380127%, loss=0.12025548808723688, time=0.8924055099487305s
Train epoch 3: top1=0.725600004196167%, top2=0.8741799592971802%, top3=0.9325999617576599%, top4=0.9648799896240234%, top5=0.9814399480819702%, l

In [14]:
append_to_file(file_name='TCL&TRL_report.txt', text=f'########################################')

# TCL & TRL from Tensorly

In [15]:
class CNN2(nn.Module):
    def __init__(self):
        super(CNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL(input_shape = (64,8,8), rank = (64,2,2))
        self.trl = TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = self.trl(x)
        return x


model2 = CNN2().to(device)


In [16]:
classifier2 = nn.Sequential(
    TCL(input_shape = (64,8,8), rank = (64,2,2)),
    TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
)

print(cp(classifier2))
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly classifier # parameters {cp(classifier2)}')

4972


In [17]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters())

In [18]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model2.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model2.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [19]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly took {end_time - start_time} time')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly had {string}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

Train epoch 1: top1=0.4334999918937683%, top2=0.6458199620246887%, top3=0.7611599564552307%, top4=0.8367999792098999%, top5=0.8924799561500549%, loss=0.19286186312556267, time=9.455854415893555s
Test epoch 1: top1=0.5182999968528748%, top2=0.724299967288971%, top3=0.828499972820282%, top4=0.8932999968528748%, top5=0.9373999834060669%, loss=0.16387201783955097, time=1.1659972667694092s
Memory Usage  - Allocated: 24.80 MB, Reserved: 70.00 MB
Train epoch 2: top1=0.5800999999046326%, top2=0.7771599888801575%, top3=0.8648999929428101%, top4=0.9203999638557434%, top5=0.9538999795913696%, loss=0.1463437400174141, time=9.616212844848633s
Test epoch 2: top1=0.6018999814987183%, top2=0.7890999913215637%, top3=0.8759999871253967%, top4=0.929099977016449%, top5=0.9603999853134155%, loss=0.13944893745183945, time=1.1717274188995361s
Train epoch 3: top1=0.6244800090789795%, top2=0.8090999722480774%, top3=0.8888999819755554%, top4=0.9344599843025208%, top5=0.9630999565124512%,

In [20]:
append_to_file(file_name='TCL&TRL_report.txt', text=f'########################################')

# TCL & TRL Method 1 just 3D tensors

In [21]:
class TCL2(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL2, self).__init__()
          #suppose it is 3d :
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)

    def forward(self, x):
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')

          return x

In [22]:
import math
class TRL2(nn.Module):
    def __init__(self, input_shape, output_shape, rank, bias = False): # last rank for the output shape
          super(TRL2, self).__init__()
          self.G = nn.Parameter(torch.empty(rank), requires_grad=True)
          nn.init.kaiming_uniform_(self.G, a=math.sqrt(5))
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)
          self.fc4 = nn.Linear(rank[3], output_shape[0], bias = bias)

    def forward(self, x):
          G = self.fc4(self.G)
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')
          x = torch.einsum('b x y z , x y z d -> b d', x, G)

          return x

In [23]:
class CNN3(nn.Module):
    def __init__(self):
        super(CNN3, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL2(input_shape = (64,8,8), rank = (64,4,4))
        self.trl = TRL2(input_shape = (64,4,4), output_shape = (10,), rank=(10,1,1,10))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = self.trl(x)
        return x


model3 = CNN3().to(device)


In [24]:
classifier3 = nn.Sequential(
    TCL2(input_shape = (64,8,8), rank = (64,2,2)),
    TRL2(input_shape = (64,2,2), output_shape = (10, ), rank=(10,1,1,10))
)

print(cp(classifier3))
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 classifier # parameters {cp(classifier3)}')

4972


In [25]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters())

In [26]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model3.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model3.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [27]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 took {end_time - start_time} time')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 had {string}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

Train epoch 1: top1=0.4143799841403961%, top2=0.630840003490448%, top3=0.7518599629402161%, top4=0.8323400020599365%, top5=0.8920199871063232%, loss=0.1967333768224716, time=9.35381007194519s
Test epoch 1: top1=0.5169999599456787%, top2=0.7249999642372131%, top3=0.8328999876976013%, top4=0.8998000025749207%, top5=0.9411999583244324%, loss=0.16595062682926653, time=1.0766716003417969s
Memory Usage  - Allocated: 24.99 MB, Reserved: 70.00 MB
Train epoch 2: top1=0.5586199760437012%, top2=0.7637999653816223%, top3=0.8585999608039856%, top4=0.9152199625968933%, top5=0.9508599638938904%, loss=0.152589133951962, time=9.260764122009277s
Test epoch 2: top1=0.5906000137329102%, top2=0.7829999923706055%, top3=0.8733999729156494%, top4=0.9282999634742737%, top5=0.9578999876976013%, loss=0.14255489587038755, time=1.0491397380828857s
Train epoch 3: top1=0.6143999695777893%, top2=0.8026999831199646%, top3=0.8882399797439575%, top4=0.9364799857139587%, top5=0.9644799828529358%, 

In [28]:
append_to_file(file_name='TCL&TRL_report.txt', text=f'########################################')